# 6. Deep Deterministic Policy Gradient (DPG)

- Deterministic Policy Gradient (DPG) is a reinforcement learning algorithm that learns a direct mapping from states to specific actions (deterministic policy) rather than a probability distribution over actions (stochastic policy)
- Benefit: it can handle continuous or infinite number of actions

In [1]:
import random
import math
import torch
from torch import nn 
from torch import optim
from collections import deque
from torch.distributions import Categorical

from frozen_lake_environment import (generate_grid_randomly,
                                     FrozenLakeEnvironment,
                                     State)
import numpy as np
from matplotlib import pyplot
from visual_utils import (render_policy_and_value, 
                          animate_policy_value_video,
                          plot_trajectory_history)
import torch.nn.functional as F
from tqdm import tqdm

In [2]:
lake_grid = [["G", "H", "F", "F"],
             ["F", "F", "F", "F"],
             ["F", "F", "H", "F"],
             ["F", "H", "S", "F"]]

reward_points = {
    "S": 0,
    "G": 10,
    "F": 0,
    "H": 0
}

frozen_lake = FrozenLakeEnvironment(grid=lake_grid,
                                    reward_points=reward_points,
                                    slippery=True)

In [28]:
class Actor(nn.Module):
    """
    Policy network π(a | s; θ).
    Outputs a categorical distribution over discrete actions.
    """
    def __init__(self, state_dim: int, action_dim: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), 
            nn.ReLU(),
            nn.Linear(hidden, hidden),  
            nn.ReLU(),
            nn.Linear(hidden, action_dim),
            nn.Tanh(), #bound action to [-1, 1]
        )

    def forward(self, x: torch.Tensor) -> Categorical:
        return self.net(x)
            
    def policy_table(self, env):
        states = [State(s_idx, env.n_cols) for s_idx in range(env.n_states)]
        policy = np.zeros(env.n_states, dtype=np.int8)
        for state in states:
            state_vec = state.get_state_feature_vec(env.n_states)
            state_vec = torch.Tensor(state_vec).unsqueeze(0) # 1 X feat_vec
            with torch.no_grad():
                action = self.forward(state_vec)
                policy[state.idx] = action.item()
        return policy

In [29]:
class Critic(nn.Module):
    """
    state-value network V(s; φ).
    Outputs one scalar per discrete action → index by chosen action.
    """
    def __init__(self, state_dim: int, action_dim: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim+action_dim, hidden), 
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, state: torch.Tensor, action:torch.Tensor) -> torch.Tensor:
        """Returns Q-values for all actions, shape (batch, act_dim)."""
        x = torch.concat([state, action], dim=1)
        return self.net(x)

In [30]:
class ReplayBuffer:
    def __init__(self, maxlen=5000):
        self.buffer = deque(maxlen=maxlen)
        
    def append(self, s, a, r, s_next, is_terminated):
        self.buffer.append((s, a, r, s_next, is_terminated))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s_next, is_terminated = map(np.array, zip(*batch))
        return s, a, r, s_next, is_terminated

    def __len__(self):
        return len(self.buffer)

In [35]:
class DDPGAgent:
    def __init__(self, env, actor, target_actor, critic, target_critic, buffer_size=5000,
                gamma=0.99, batch_size=64, tau=0.001, lr_actor=0.001, lr_critic=0.001):
        self.env = env
        self.actor = actor
        self.target_actor = target_actor
        
        self.critic = critic  
        self.target_critic = target_critic                                       
        
        self.opt_actor  = optim.Adam(self.actor.parameters(),  lr=lr_actor)
        self.opt_critic = optim.Adam(self.critic.parameters(), lr=lr_critic)

        self.replay_buffer = ReplayBuffer(buffer_size)
        self.gamma = gamma
        self.batch_size = batch_size
        self.tau = tau

    def select_action(self, state, noise=0.1):
        with torch.no_grad():   
            state = torch.FloatTensor(state).unsqueeze(0)
        
            action = self.actor(state).squeeze(0).numpy()

        # add exporation noise
        action += torch.normal(mean=0, std=0.1, size=action.shape)
        return torch.clip(action, -1, 1)

    def critic_update(self, s, r, next_s, next_a, mask):
        with torch.no_grad():
            next_a = self.target_actor(next_s)
            Q_next = self.target_critic(next_s, next_a).squeeze(-1)                 
            target = r + self.gamma * Q_next * mask   

        Q = self.critic(s).squeeze(-1)
        critic_loss = F.mse_loss(Q, target)
        self.opt_critic.zero_grad()
        critic_loss.backward()
        nn.utils.clip_grad_norm_(self.critic.parameters(), 1.0)
        self.opt_critic.step()
        return critic_loss

    def actor_update(self, s):
        actor_loss = -self.critic(s, self.actor(s)).mean()
        
        self.opt_actor.zero_grad()
        actor_loss.backward()
        nn.utils.clip_grad_norm_(self.critic.parameters(), 1.0)
        self.opt_actor.step()
        return actor_loss

    def soft_update(self, target, source):
        for target_param, src_param in zip(target.parameters(), source.parameters()):
            target_param.data.copy_(
                self.tau * src_param.data + (1.0 - self.tau) * target_param.data
            )
            
    def update(self):
        if len(self.replay_buffer) < self.batch_size:
            return None

        s, a, r, next_s, is_terminated = self.replay_buffer.sample(self.batch_size)
        
        s      = torch.FloatTensor(s).unsqueeze(0)
        a      = torch.LongTensor([a])
        r      = torch.FloatTensor([r])
        next_s = torch.FloatTensor(next_s).unsqueeze(0)
        next_a = torch.LongTensor([next_a])
        
        mask   = torch.FloatTensor([0.0 if is_terminated else 1.0])

        critic_loss = self.critic_update(s, a, r, next_s)
        actor_loss = self.actor_update(s)

        # soft update
        self.soft_update(target_actor, actor)
        self.soft_update(target_critic, critic)
        
        return critic_loss, actor_loss

In [32]:
def train(env, n_episodes, gamma=0.99, lr_actor=1e-4, lr_critic=1e-4):
    actor = Actor(env.n_states, action_dim=1)
    target_actor = Actor(env.n_states, action_dim=1)
    
    critic = Critic(env.n_states, action_dim=1)
    target_critic = Critic(env.n_states, action_dim=1)
    
    agent = DDPGAgent(env, actor, target_actor, critic, target_critic, 
                      lr_actor=lr_actor, lr_critic=lr_critic, gamma=gamma)
    
    trajectory_histories = []
    policy_histories = []
    V_histories = []
    for episode_idx in range(n_episodes):
        s = env.find('S')
        s_feat = s.get_state_feature_vec(env.n_states)
        a = agent.select_action(s_feat)
        trajectory = []
        G  = 0
        
        while True:
            trajectory.append(s.coord)
            result = env.step(s, a)
            r = result["reward"]            
            next_s = result["new_state"]
            next_s_feat = next_s.get_state_feature_vec(env.n_states)
            is_terminated = result["is_terminated"]

            agent.replay_buffer.append(s_feat,
                                 a,
                                 r,
                                 next_s_feat,
                                 is_terminated)
    
            c_loss, a_loss = agent.update()
            
            s = next_s
            s_feat = next_s_feat
            a = agent.select_action(s_feat)

            G += r
            if is_terminated:
                break

        trajectory_histories.append(trajectory)
        
        if episode_idx % 500 == 0:
            print("critic loss: ", c_loss, " actor loss: ", a_loss)
            print("Return: ", G)
            
            policy = agent.actor.policy_table(env)
            policy_histories.append(policy)
     
    return policy_histories, trajectory_histories

In [33]:
# lake_grid = [["G", "H", "F", "F"],
#              ["F", "F", "F", "F"],
#              ["F", "F", "H", "F"],
#              ["F", "H", "S", "F"]]

lake_grid = [["F", "F", "S", "F", "H"],
             ["F", "F", "H", "F", "F"],
             ["F", "F", "F", "G", "F"],
             ["F", "H", "F", "F", "F"],
             ["H", "H", "F", "F", "F"]]

reward_points = {
    "S": 0,
    "G": 1,
    "F": 0,
    "H": 0
}

frozen_lake = FrozenLakeEnvironment(grid=lake_grid,
                                    reward_points=reward_points,
                                    slippery=True)

In [34]:
policy_histories, v_histories, trajectory_histories = train(frozen_lake, n_episodes=100)

TypeError: linear(): argument 'input' (position 1) must be Tensor, not numpy.ndarray